In [16]:
# Program to see BGP pattern and BGP-based DDoS scrubbing for DDoS attack on Github on 2018
# Github AS36459, Akamai prolexic AS32787
# DateTime: 28 Feb, 2018, 9:15 AM PST (5:15 PM) , Service restored in 9:30 AM PST
# Prefix under attack 192.30.253.0/24

# Note: All the helper methods (e.g. detect_ip_version) are at the bottom of the program file
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import csv 
import datetime

scrubber_asn = "32787"
stream = pybgpstream.BGPStream(
    from_time="2018-02-28 15:00:00 UTC",
    until_time="2018-02-28 18:30:00 UTC",
    record_type="updates",
#     collectors=["rrc00", "rrc03", "rrc25", "route-views.amsix"],
    filter = "path 32787$" #Look for all the prefixes that contain AS200020 as an immediate provider
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
# stream.add_filter('elemtype', 'announcements')


prefix_details = [] # For storing ipv4 prefixes

print("Starting")
# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    collector = rec.collector
    for elem in rec:
        peer_asn = elem.peer_asn
        peer_ip = elem.peer_address
        
        pfx = elem.fields["prefix"]
        ip = ipaddress.ip_network(pfx)
        pfx_len = ip.prefixlen
        
        # Find upstream ASN in an AS path
        as_path = elem.fields["as-path"]
        as_path = as_path.split()
        orig = as_path[-1]
        
        # Call a function to find provider because provider might not always be the second last AS in case of
        # AS path prepending and sibling ASes
#         second_as = find_immediate_provider(as_path)
        
        elem_type = elem.type

        # Store origin asn and announcement time in a dictionary
        asn_time = {}
        
        # Condition that it is a scrub AS 
        # 1. AS should be the second last AS in an AS path
        # See all the prefixes that has AS200020 as a second last hop and only IPv4
        if detect_ip_version(pfx) == 'IPv4':
#             print("Prefix %s, Origin %s , time %s elem %s" %(pfx, orig, time, elem))
            asn_time["origin"] = orig
    
            # Convert to UTC time
            time_utc = datetime.datetime.utcfromtimestamp(time)

            # Convert UTC datetime to string
            time_utc_str = time_utc.strftime("%Y-%m-%d %H:%M:%S")
            
            asn_time["time"] = time
            asn_time["time_utc_str"] = time_utc_str
            
            asn_time["peer_asn"] = peer_asn
            asn_time["peer_ip"] = peer_ip
#             asn_time["elem_type"] = elem_type
#             asn_time["collector"] = collector
            asn_time["prefix"] = pfx
        
#             asn_time["second_as"] = second_as
            asn_time["type"] = elem_type
            asn_time["as_path"] = as_path
       

            prefix_details.append(asn_time)                   
print("Completed")

# Store the results into a csv file
csv_file = "/home/shyam/jupy/scrubber_activation/data/as32787_28_Feb_2018_rrc00.csv"

# Write to CSV
with open(csv_file, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=prefix_details[0].keys())
    writer.writeheader()
    writer.writerows(prefix_details)

print(f"Data has been written to {csv_file}")

Starting
Completed
Data has been written to /home/shyam/jupy/scrubber_activation/data/as32787_28_Feb_2018_rrc00.csv


In [10]:
# Program to see BGP pattern and BGP-based DDoS scrubbing for DDoS attack on Github on 2018, same as above 
# but this program looks into BGP withdrawals

# Note: All the helper methods (e.g. detect_ip_version) are at the bottom of the program file
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import csv 
import datetime

scrubber_asn = "32787"
prefix = "192.30.253.0/24"

stream = pybgpstream.BGPStream(
    from_time="2018-02-28 17:00:00 UTC",
    until_time="2018-02-28 18:30:00 UTC",
    collectors=["rrc00"],# "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates" ,
    filter="prefix less "+prefix
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('elemtype', 'withdrawals')


prefix_details = [] # For storing ipv4 prefixes

print("Starting")
# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    collector = rec.collector
    for elem in rec:
        print(elem)
        peer_asn = elem.peer_asn
        peer_ip = elem.peer_address
        
        pfx = elem.fields["prefix"]
        ip = ipaddress.ip_network(pfx)
        pfx_len = ip.prefixlen
        
        # Store origin asn and announcement time in a dictionary
        asn_time = {}
        
        # Condition that it is a scrub AS 
        # 1. AS should be the second last AS in an AS path
        # See all the prefixes that has AS200020 as a second last hop and only IPv4
        if detect_ip_version(pfx) == 'IPv4':
            # Convert to UTC time
            time_utc = datetime.datetime.utcfromtimestamp(time)

            # Convert UTC datetime to string
            time_utc_str = time_utc.strftime("%Y-%m-%d %H:%M:%S")
            
            asn_time["time"] = time
            asn_time["time_utc_str"] = time_utc_str
            
            asn_time["peer_asn"] = peer_asn
            asn_time["peer_ip"] = peer_ip
#             asn_time["elem_type"] = elem_type
#             asn_time["collector"] = collector
            asn_time["prefix"] = pfx
            prefix_details.append(asn_time)                   
print("Completed")

# Store the results into a csv file
csv_file = "/home/shyam/jupy/scrubber_activation/data/as36459_28_Feb_2018_rrc00_withdrawal.csv"

# Write to CSV
with open(csv_file, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=prefix_details[0].keys())
    writer.writeheader()
    writer.writerows(prefix_details)

print(f"Data has been written to {csv_file}")

Starting
update|W|1519838855.000000|ris|rrc00|None|None|7018|12.0.1.63|192.30.252.0/23|None|None|None|None|None
update|W|1519838855.000000|ris|rrc00|None|None|7018|12.0.1.63|192.30.252.0/22|None|None|None|None|None
update|W|1519838855.000000|ris|rrc00|None|None|7018|12.0.1.63|192.30.253.0/24|None|None|None|None|None
Completed
Data has been written to /home/shyam/jupy/scrubber_activation/data/as36459_28_Feb_2018_rrc00_withdrawal.csv


In [3]:
# Detects if a given string is an IPv4 or IPv6
import ipaddress

def detect_ip_version(ip_network_str):
    try:
        ip_network = ipaddress.ip_network(ip_network_str, strict=False)
        if isinstance(ip_network, ipaddress.IPv4Network):
            return "IPv4"
        elif isinstance(ip_network, ipaddress.IPv6Network):
            return "IPv6"
    except ValueError:
        return "Invalid IP address or network"

In [4]:
# Program to find upstream provider using condition: 
# a. If the same origin AS is prepending, upstream AS is the next one after prepending. 
# b. If the origin AS has siblings, remove siblings. 
# CAVEAT: This program does not check as_path = [200, 300, 400, 400, 8074, 8075, 8075] where there is repeatation of 
# new ASes other than siblings

# Function to call an API to get the list of siblings for a given ASN (last origin)
def api_get_siblings(asn):
    
    as_rank = AsRank()
    siblings = as_rank.get_all_siblings(asn)

    return siblings[1] # It contains list of total siblings count and list of ASNs. We are concerned only with the latter.

# Function to find the immediate provider ASN
def find_immediate_provider(as_path):
    if len(as_path) < 2:
        return None  # Not enough ASNs in path to determine provider

    last_origin_asn = as_path[-1]  # The last ASN is the origin ASN

    # Check for sequentially repeated ASNs
    repeated_asn = None
    for i in range(len(as_path) - 1, 0, -1):
        if as_path[i-1] == as_path[i]:
            repeated_asn = as_path[i]
        else:
            # If we find an ASN that is not the same as the repeated ASN,
            # and we have found a repeated ASN, return the one before the repeated ASN
            if repeated_asn is not None:
                return as_path[i-1]  # This is the upstream provider
            break

# Do this check to call API only in case an origin has mutliple siblings

# If no repeated ASN found or it's the only ASN in the path
#     if repeated_asn is None:
#         # Retrieve siblings for the last ASN (last origin in the AS path)
#         try:
#             print("AS rank API for getting siblings")
#             sibling_list = api_get_siblings(last_origin_asn)
#         except Exception as e:
#             print(f"Error retrieving siblings: {e}")
#             return None  # Handle API failure appropriately

#         # Traverse the AS path 
#         for i in range(len(as_path) - 1, 0, -1):
#             if str(as_path[i-1]) not in sibling_list:
#                 return as_path[i-1]  # The first non-sibling ASN is the upstream provider

    return as_path[1]  # If all checks fail, return the second ASN by default


# Test cases based on your examples
# as_path = [400, 100, 300, 300]  # Case 1: No repetition, no siblings
# # as_path = ["200", "300", "400", "400", "6584", "8074", "8075"]  # Case 2: Sequentially repeated, should return 300


# # # Finding immediate providers
# print("Immediate provider for AS path:", find_immediate_provider(as_path))  # Expected Output: 300


In [ ]:
# Method to get siblings of an AS using AS rank API
# This part of the code is used from https://github.com/bgpkit/pyasrank

# MIT License

# Copyright (c) 2021 Mingwei Zhang

# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:

# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.

# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.
import json
import logging
from datetime import datetime

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ASRANK_ENDPOINT = "https://api.asrank.caida.org/v2/graphql"


def ts_to_date_str(ts):
    """
    Convert timestamp to a date. This is used for ASRank API which only takes
    date strings with no time as parameters.api
    """
    return datetime.utcfromtimestamp(int(ts)).strftime("%Y-%m-%d")


class AsRank:
    """
    Utilities for using ASRank services
    """

    def __init__(self, max_ts=""):
        self.data_ts = None

        # various caches to avoid duplicate queries
        self.cache = None
        self.cone_cache = None
        self.neighbors_cache = None
        self.siblings_cache = None
        self.organization_cache = None

        self.queries_sent = 0

        self.session = None
        self._initialize_session()

        self.init_cache(max_ts)

    def _initialize_session(self):
        self.session = requests.Session()
        retries = Retry(total=5,
                        backoff_factor=1,
                        status_forcelist=[500, 502, 503, 504])
        self.session.mount(ASRANK_ENDPOINT, HTTPAdapter(max_retries=retries))

    def _close_session(self):
        if self.session:
            self.session.close()

    def _send_request(self, query):
        """
        send requests to ASRank endpoint
        :param query:
        :return:
        """

        r = self.session.post(url=ASRANK_ENDPOINT, json={'query': query})
        r.raise_for_status()
        self.queries_sent += 1
        return r

    def init_cache(self, ts):
        """
        Initialize the ASRank cache for the timestamp ts
        :param ts:
        :return:
        """
        self.cache = {}
        self.cone_cache = {}
        self.neighbors_cache = {}
        self.siblings_cache = {}
        self.organization_cache = {}
        self.queries_sent = 0
        if isinstance(ts, int):
            ts = ts_to_date_str(ts)

        ####
        # Try to cache datasets available before the given ts
        ####
        graphql_query = """
            {
              datasets(dateStart:"2000-01-01", dateEnd:"%s", sort:"-date", first:1){
                edges {
                  node {
                    date
                  }
                }
              }
            }
        """ % ts
        r = self._send_request(graphql_query)

        edges = r.json()['data']['datasets']['edges']
        if edges:
            self.data_ts = edges[0]["node"]["date"]
            return

        # if code reaches here, we have not found any datasets before ts. we should now try to find one after ts.
        # this is the best effort results
        logging.warning("cannot find dataset before date %s, looking for the closest one after it now" % ts)

        graphql_query = """
            {
              datasets(dateStart:"%s", sort:"date", first:1){
                edges {
                  node {
                    date
                  }
                }
              }
            }
        """ % ts
        r = self._send_request(graphql_query)
        edges = r.json()['data']['datasets']['edges']
        if edges:
            self.data_ts = edges[0]["node"]["date"]
            logging.warning("found closest dataset date to be %s" % self.data_ts)
            return
        else:
            raise ValueError("no datasets from ASRank available to use for tagging")

    def _query_asrank_for_asns(self, asns, chunk_size=100):
        asns = [str(asn) for asn in asns]
        asns_needed = [asn for asn in asns if asn not in self.cache]
        if not asns_needed:
            return

        # https://stackoverflow.com/a/312464/768793
        def chunks(lst, n):
            """Yield successive n-sized chunks from lst."""
            for i in range(0, len(lst), n):
                yield lst[i:i + n]

        for asns in chunks(asns_needed, chunk_size):

            graphql_query = """
                {
                  asns(asns: %s, dateStart: "%s", dateEnd: "%s", first:%d, sort:"-date") {
                    edges {
                      node {
                        date
                        asn
                        asnName
                        rank
                        organization{
                          country{
                            iso
                            name
                          }
                          orgName
                          orgId
                        } asnDegree {
                          provider
                          peer
                          customer
                          total
                          transit
                          sibling
                        }
                      }
                    }
                  }
                }
            """ % (json.dumps(asns), self.data_ts, self.data_ts, len(asns))
            r = self._send_request(graphql_query)
            try:
                for node in r.json()['data']['asns']['edges']:
                    data = node['node']
                    if data['asn'] not in self.cache:
                        if "asnDegree" in data:
                            degree = data["asnDegree"]
                            degree["provider"] = degree["provider"] or 0
                            degree["customer"] = degree["customer"] or 0
                            degree["peer"] = degree["peer"] or 0
                            degree["sibling"] = degree["sibling"] or 0
                            data["asnDegree"] = degree
                        self.cache[data['asn']] = data
                for asn in asns:
                    if asn not in self.cache:
                        self.cache[asn] = None
            except KeyError as e:
                logging.error("Error in node: {}".format(r.json()))
                logging.error("Request: {}".format(graphql_query))
                raise e

    

    def get_all_siblings(self, asn, skip_asrank_call=False):
        """
        get all siblings for an ASN
        :param asn: AS number to query for all siblings
        :param skip_asrank_call: skip asrank call if already done
        :return: a tuple of (TOTAL_COUNT, ASNs)
        """
        # FIXME: pagination does not work here. Example ASN5313.
        asn = str(asn)
        if asn in self.siblings_cache:
            return self.siblings_cache[asn]

        if not skip_asrank_call:
            self._query_asrank_for_asns([asn])

        if asn not in self.cache or self.cache[asn] is None:
            return 0, []
        asrank_info = self.cache[asn]
        if "organization" not in asrank_info or asrank_info["organization"] is None:
            return 0, []

        org_id = self.cache[asn]["organization"]["orgId"]

        if org_id in self.organization_cache:
            data = self.organization_cache[org_id]
        else:
            graphql_query = """
            {
            organization(orgId:"%s"){
              orgId,
              orgName,
              members{
                numberAsns,
                numberAsnsSeen,
                asns{totalCount,edges{node{asn,asnName}}}
              }
            }}        
            """ % org_id
            r = self._send_request(graphql_query)
            data = r.json()["data"]["organization"]
            self.organization_cache[org_id] = data

        if data is None:
            return 0, []

        total_cnt = data["members"]["asns"]["totalCount"]
        siblings = set()
        for sibling_data in data["members"]["asns"]["edges"]:
            siblings.add(sibling_data["node"]["asn"])
        if asn in siblings:
            siblings.remove(asn)
            total_cnt -= 1

        # NOTE: this assert can be wrong when number of siblings needs pagination
        # assert len(siblings) == total_cnt - 1

        siblings = list(siblings)
        self.neighbors_cache[asn] = (total_cnt, siblings)
        return total_cnt, siblings